# Import libraries

In [1]:
import os
import shutil
import pandas as pd
import subprocess
from tqdm import tqdm
import concurrent.futures

import sys
sys.path.append('..')
from utils.audio_util import validate_wav_files, convert_wav_to_flac, resample_audios, trim_silence_with_vad, normalize_audio_files
from utils.file_util import recursive_copy

/home/pruuwu/dubbing-ai/Restructure/converter/../utils/audio_util.py:12: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import SepformerSeparation as separator


# Convert invalid

In [2]:
# Make sure wav is valid
validate_wav_files("../data/raw/LJSpeech-1.1/wavs")

Found 13100 WAV files to validate


Validating WAV files: 100%|██████████| 13100/13100 [08:52<00:00, 24.60it/s]

Successfully validated 13100 WAV files


13100

# Read metadata

In [3]:
DEST_DIR = "../data/converted/LJSpeech-to-vctk"
DEST_TEXT_PATH = os.path.join(DEST_DIR, "txt/LJSpeech")
DEST_AUDIO_PATH = os.path.join(DEST_DIR, "wav44/LJSpeech")
SRC_PATH = '../data/raw/LJSpeech-1.1'

# Clean and create directories
if os.path.exists(DEST_DIR):
   print("Clearing destination folder")
   shutil.rmtree(DEST_DIR)

os.makedirs(DEST_TEXT_PATH, exist_ok=True)
os.makedirs(DEST_AUDIO_PATH, exist_ok=True)

file_path = os.path.join(SRC_PATH, "metadata.csv")

if os.path.exists(file_path):  # Check if file exists
    df = pd.read_csv(
        file_path,
        sep="|",
        header=None,
        names=["ID", "Transcription", "Normalized Transcription"],
        quoting=3
    )
else:
    print(f"Error: File not found at {file_path}")

In [4]:
df.iloc[1193]

ID                                                                 LJ005-0074
Transcription                               cap. 64, and 5 George IV. cap. 85
Normalized Transcription    cap. sixty-four, and five George the fourth ca...
Name: 1193, dtype: object

In [5]:
df.iloc[1196]["Normalized Transcription"]

'"expedient to introduce such measures and arrangements as shall not only provide for the safe custody,'

# Moving files to the new directory

In [6]:
def process_file(item):
    """Process a single WAV file."""
    w, count, transcription = item
    src_wav_file = os.path.join(SRC_WAV_PATH, w)
    dest_wav_file = os.path.join(DEST_AUDIO_PATH, f"LJSpeech_{count:03d}_mic1.flac")
    dest_txt_file = os.path.join(DEST_TEXT_PATH, f"LJSpeech_{count:03d}.txt")
    
    try:
        # Write text file
        with open(dest_txt_file, "w", encoding="utf-8") as f:
            f.write(transcription)
        
        # Convert audio file
        convert_wav_to_flac(src_wav_file, dest_wav_file)
        
        return True
    except Exception as e:
        print(f"Error processing {w}: {str(e)}")
        return False

SRC_WAV_PATH = os.path.join(SRC_PATH, 'wavs')

if not os.path.exists(SRC_WAV_PATH):
    print(f"Error: Directory not found at {SRC_WAV_PATH}")
else:
    # Create a lookup dictionary for faster access
    id_to_transcription = dict(zip(df["ID"], df["Normalized Transcription"])) # Use normalize to make sure it not contain numeric character like 0 1 2
    
    list_wavs = os.listdir(SRC_WAV_PATH)
    
    # Prepare the items to process
    items_to_process = []
    count = 1
    
    for w in list_wavs:
        txt_name = os.path.splitext(w)[0]
        if txt_name in id_to_transcription:
            items_to_process.append((w, count, id_to_transcription[txt_name]))
            count += 1
        else:
            print(f"Error: {txt_name} not found in dataset")
    
    # Use ThreadPoolExecutor for parallel processing
    with concurrent.futures.ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
        # Use tqdm to show progress
        list(tqdm(
            executor.map(process_file, items_to_process),
            total=len(items_to_process),
            desc="Processing audio files"
        ))

Processing audio files: 100%|██████████| 13100/13100 [01:09<00:00, 188.18it/s]


# Resample, trim, and normalize audio

In [7]:
# Create destination directory if it doesn't exist
os.makedirs("../data/converted/LJSpeech-to-vctk/wav16_silence_trimmed", exist_ok=True)

# Copy all files from wav44 to wav16_silence_trimmed
src_dir = "../data/converted/LJSpeech-to-vctk/wav44"
dst_dir = "../data/converted/LJSpeech-to-vctk/wav16_silence_trimmed"

recursive_copy(src_dir, dst_dir)

In [8]:
# Resample all files in wav16_silence_trimmed to 16kHz
SAMPLE_RATE = 16000
NUM_RESAMPLE_THREADS = 8

resample_audios(
  input_folders=dst_dir,
  file_ext="flac",
  sample_rate=SAMPLE_RATE,
  n_jobs=NUM_RESAMPLE_THREADS
)

Resampling the audio files...
Found 13100 files...


100%|██████████| 13100/13100 [00:16<00:00, 788.87it/s]

Done !


In [9]:
# Trim silence at the beginning and end of each audio file
trim_silence_with_vad(
  input_folder=dst_dir,
  file_extension="flac",
)

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /home/pruuwu/.cache/torch/hub/master.zip


Found 13100 .flac files to process


Processing files: 100%|██████████| 13100/13100 [22:41<00:00,  9.62it/s]


Processing complete


In [10]:
# Normalize the volume of all audio files to -27dB
normalize_audio_files(
    input_dir=dst_dir,
)

Normalizing audio files: 100%|██████████| 13100/13100 [45:40<00:00,  4.78it/s]


# Output stat

In [11]:
import os
import glob
import soundfile as sf
from tqdm import tqdm

# Path pattern
path_pattern = "../data/converted/LJSpeech-to-vctk/wav16_silence_trimmed/**/*.flac"

# Get all flac files matching the pattern
flac_files = glob.glob(path_pattern, recursive=True)

if not flac_files:
    print(f"No FLAC files found matching pattern: {path_pattern}")
else:
    print(f"Found {len(flac_files)} FLAC files. Processing...")
    
    # Calculate total duration
    total_duration_seconds = 0
    
    # Track speaker folders
    speaker_folders = set()
    
    # Extract base directory for later use in calculating speaker directories
    base_dir = os.path.normpath("../data/converted/LJSpeech-to-vctk/wav16_silence_trimmed")
    
    # Use tqdm for progress bar
    for flac_file in tqdm(flac_files):
        try:
            # Get audio info
            info = sf.info(flac_file)
            total_duration_seconds += info.duration
            
            # Extract speaker folder - take the directory right after wav16_silence_trimmed/
            rel_path = os.path.relpath(os.path.dirname(flac_file), base_dir)
            if '/' in rel_path:
                speaker = rel_path.split('/')[0]  # First directory is the speaker
            else:
                speaker = rel_path  # If there's no further nesting
                
            speaker_folders.add(speaker)
            
        except Exception as e:
            print(f"Error processing {flac_file}: {e}")
    
    # Convert to hours
    total_duration_hours = total_duration_seconds / 3600
    
    # Print results
    print(f"\nTotal duration: {total_duration_hours:.2f} hours")
    print(f"                ({total_duration_hours*60:.2f} minutes)")
    print(f"                ({total_duration_hours*3600:.2f} seconds)")
    
    # Print only the number of speakers
    print(f"Number of speakers: {len(speaker_folders)}")

Found 13100 FLAC files. Processing...


100%|██████████| 13100/13100 [00:00<00:00, 15011.54it/s]


Total duration: 23.79 hours
                (1427.34 minutes)
                (85640.29 seconds)
Number of speakers: 1
